In [1]:
# ================= INSTALL LATEST XGBOOST =================
!pip install -U xgboost --quiet
import xgboost
print('XGBoost Version After Install:', xgboost.__version__)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.7/131.7 MB 14.5 MB/s eta 0:00:0000:0100:01
XGBoost Version After Install: 3.2.0


In [2]:
# ================= GPU VERIFICATION =================
import torch, xgboost
print('Torch CUDA Available:', torch.cuda.is_available())
print('XGBoost Version:', xgboost.__version__)


Torch CUDA Available: True
XGBoost Version: 3.2.0


In [3]:
# ================= GPU VERIFICATION CELL =================
import os
import torch
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Name:', torch.cuda.get_device_name(0))

# For XGBoost (latest Kaggle versions)
xgb_gpu_params = {'device': 'cuda', 'tree_method': 'hist'}

# For LightGBM
lgbm_gpu_params = {'device': 'gpu', 'gpu_platform_id': 0, 'gpu_device_id': 0}

# For CatBoost
cat_gpu_params = {'task_type': 'GPU', 'devices': '0'}
print('GPU parameters prepared successfully.')


CUDA Available: True
GPU Name: Tesla T4
GPU parameters prepared successfully.


In [4]:
import pandas as pd
import numpy as np
import optuna
import lightgbm as lgb
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.utils.class_weight import compute_class_weight
N_SPLITS = 5   # keep 5 for tuning (faster)
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)


In [5]:
# LOAD DATA
train = pd.read_csv('/kaggle/input/competitions/playground-series-s6e2/train.csv')
test = pd.read_csv('/kaggle/input/competitions/playground-series-s6e2/test.csv')

test_ids = test['id'].copy()
train.drop(columns=['id'], inplace=True)
test.drop(columns=['id'], inplace=True)

target = train.columns[-1]
y = train[target]
X = train.drop(columns=[target])

if y.dtype == object:
    y = y.map({'Absence':0,'Presence':1})
y = y.astype(int)

# ================= DEFINE SCALE_POS_WEIGHT =================
neg = (y == 0).sum()
pos = (y == 1).sum()
scale_pos_weight = neg / pos

print("Negatives:", neg)
print("Positives:", pos)
print("Scale Pos Weight:", scale_pos_weight)

X = X.apply(lambda col: pd.to_numeric(col, errors='coerce')).fillna(0)
test = test.apply(lambda col: pd.to_numeric(col, errors='coerce')).fillna(0)


Negatives: 347546
Positives: 282454
Scale Pos Weight: 1.2304516841680415


In [ ]:

# ================= ADVANCED INTERACTION FEATURES =================

X["Angina_HR"] = X["Exercise angina"] * X["Max HR"]
test["Angina_HR"] = test["Exercise angina"] * test["Max HR"]

X["ST_Age"] = X["ST depression"] * X["Age"]
test["ST_Age"] = test["ST depression"] * test["Age"]

X["HR_BP_ratio"] = X["Max HR"] / (X["BP"] + 1)
test["HR_BP_ratio"] = test["Max HR"] / (test["BP"] + 1)

X["Chol_Age_ratio"] = X["Cholesterol"] / (X["Age"] + 1)
test["Chol_Age_ratio"] = test["Cholesterol"] / (test["Age"] + 1)


In [ ]:

# ================= OOF TARGET ENCODING =================

from sklearn.model_selection import StratifiedKFold
import numpy as np

cat_cols = ["Chest pain type", "EKG results", "Slope of ST", "Thallium"]

skf_te = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for col in cat_cols:
    oof_encoded = np.zeros(len(X))
    
    for train_idx, val_idx in skf_te.split(X, y):
        X_train_fold, X_val_fold = X.iloc[train_idx], X.iloc[val_idx]
        y_train_fold = y.iloc[train_idx]
        
        target_means = y_train_fold.groupby(X_train_fold[col]).mean()
        oof_encoded[val_idx] = X_val_fold[col].map(target_means)
    
    global_means = y.groupby(X[col]).mean()
    test_encoded = test[col].map(global_means)
    
    X[col + "_TE"] = oof_encoded
    test[col + "_TE"] = test_encoded


In [6]:

# ================= FEATURE ENGINEERING =================

# -------- Age Features --------
X["Age_squared"] = X["Age"] ** 2
test["Age_squared"] = test["Age"] ** 2

X["Age_BP"] = X["Age"] * X["BP"]
test["Age_BP"] = test["Age"] * test["BP"]

X["Age_Chol"] = X["Age"] * X["Cholesterol"]
test["Age_Chol"] = test["Age"] * test["Cholesterol"]

# -------- Ratios --------
X["BP_Age_ratio"] = X["BP"] / (X["Age"] + 1)
test["BP_Age_ratio"] = test["BP"] / (test["Age"] + 1)

X["Chol_BP_ratio"] = X["Cholesterol"] / (X["BP"] + 1)
test["Chol_BP_ratio"] = test["Cholesterol"] / (test["BP"] + 1)

# -------- Heart Stress --------
X["HR_Stress_Index"] = X["Max HR"] / (220 - X["Age"])
test["HR_Stress_Index"] = test["Max HR"] / (220 - test["Age"])

# -------- ST Engineering --------
X["ST_Slope_Interaction"] = X["ST depression"] * X["Slope of ST"]
test["ST_Slope_Interaction"] = test["ST depression"] * test["Slope of ST"]

X["Severe_ST"] = (X["ST depression"] > 2).astype(int)
test["Severe_ST"] = (test["ST depression"] > 2).astype(int)

# -------- Vessel Interaction --------
X["Vessel_Thallium"] = X["Number of vessels fluro"] * X["Thallium"]
test["Vessel_Thallium"] = test["Number of vessels fluro"] * test["Thallium"]

# -------- Clinical Risk Score --------
risk_cols = ["FBS over 120", "Exercise angina"]
X["Risk_Score"] = X[risk_cols].sum(axis=1) + X["Severe_ST"]
test["Risk_Score"] = test[risk_cols].sum(axis=1) + test["Severe_ST"]


In [7]:
# ================= OPTUNA XGBOOST (LATEST 2.x API) =================
def tune_xgb(trial):
    params = {
        'n_estimators': 6000,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.5),
        'max_depth': trial.suggest_int('max_depth', 1, 15),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 12),
        'gamma': trial.suggest_float('gamma', 0.0, 1.0),
        'subsample': trial.suggest_float('subsample', 0.1, 1),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.1, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 10),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 10),
        'tree_method': 'hist',
        'device': 'cuda',
        'eval_metric': 'auc',
        'early_stopping_rounds': 200,
        'random_state': 42,
        'scale_pos_weight': scale_pos_weight
    }

    oof = np.zeros(len(X))

    for tr, va in skf.split(X, y):
        model = XGBClassifier(**params)
        model.fit(
            X.iloc[tr], y.iloc[tr],
            eval_set=[(X.iloc[va], y.iloc[va])],
            verbose=False
        )
        oof[va] = model.predict_proba(X.iloc[va])[:, 1]

    return roc_auc_score(y, oof)


In [11]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 630000 entries, 0 to 629999
Data columns (total 14 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   Age                      630000 non-null  int64  
 1   Sex                      630000 non-null  int64  
 2   Chest pain type          630000 non-null  int64  
 3   BP                       630000 non-null  int64  
 4   Cholesterol              630000 non-null  int64  
 5   FBS over 120             630000 non-null  int64  
 6   EKG results              630000 non-null  int64  
 7   Max HR                   630000 non-null  int64  
 8   Exercise angina          630000 non-null  int64  
 9   ST depression            630000 non-null  float64
 10  Slope of ST              630000 non-null  int64  
 11  Number of vessels fluro  630000 non-null  int64  
 12  Thallium                 630000 non-null  int64  
 13  Heart Disease            630000 non-null  object 
dtypes: f

In [8]:
study_xgb = optuna.create_study(direction='maximize')
study_xgb.optimize(tune_xgb, n_trials=30)

best_xgb_params = study_xgb.best_params
print('Best XGB Params:', best_xgb_params)


[I 2026-02-21 11:15:12,971] A new study created in memory with name: no-name-ab699cff-9942-4d07-85a6-c2c408ac857b
/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [11:15:16] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)
[I 2026-02-21 11:15:27,163] Trial 0 finished with value: 0.9552799119455938 and parameters: {'learning_rate': 0.4702827054242329, 'max_depth': 2, 'min_child_weight': 2, 'gamma': 0.6011606084337436, 'subsample': 0.7669412210230342, 'colsample_bytree': 0.6176994452579511, 'reg_alpha': 5.191053530650099, 'reg_lambda': 7.953657013233

Best XGB Params: {'learning_rate': 0.08677943951193733, 'max_depth': 2, 'min_child_weight': 2, 'gamma': 0.020954419156370918, 'subsample': 0.930879759127863, 'colsample_bytree': 0.5555685557917291, 'reg_alpha': 3.6696709935395035, 'reg_lambda': 7.5948872214397385}


In [9]:
# ================= FINAL OPTIMIZED STACK (STABLE VERSION) =================

import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# -------- Ensure scale_pos_weight exists --------
if 'scale_pos_weight' not in globals():
    neg = (y == 0).sum()
    pos = (y == 1).sum()
    scale_pos_weight = neg / pos

# -------- Ensure best_xgb_params exists --------
if 'best_xgb_params' not in globals():
    best_xgb_params = {}

# Remove conflicting keys
for key in ['tree_method', 'device', 'eval_metric', 'early_stopping_rounds']:
    best_xgb_params.pop(key, None)

# -------- 10 Fold CV --------
N_SPLITS = 10
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

# -------- XGB Params --------
xgb_params = {
    **best_xgb_params,
    'learning_rate': 0.02,
    'n_estimators': 4000,
    'tree_method': 'hist',
    'device': 'cuda',
    'eval_metric': 'auc',
    'random_state': 42,
    'scale_pos_weight': scale_pos_weight * 1.05,
    'early_stopping_rounds': 200
}

# -------- LGB Params --------
lgb_params = {
    'n_estimators': 4000,
    'learning_rate': 0.02,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.5,
    'reg_lambda': 2.0,
    'random_state': 42
}

oof = np.zeros((len(X), 2))
test_preds = np.zeros((len(test), 2))

for fold, (tr, va) in enumerate(skf.split(X, y)):
    print(f'Fold {fold+1}')

    # XGBoost
    xgb_model = XGBClassifier(**xgb_params)

    xgb_model.fit(
        X.iloc[tr], y.iloc[tr],
        eval_set=[(X.iloc[va], y.iloc[va])],
        verbose=False
    )

    best_iter = xgb_model.get_booster().best_iteration
    if best_iter is None:
        best_iter = xgb_model.n_estimators

    oof[va, 0] = xgb_model.predict_proba(
        X.iloc[va],
        iteration_range=(0, best_iter)
    )[:, 1]

    test_preds[:, 0] += xgb_model.predict_proba(
        test,
        iteration_range=(0, best_iter)
    )[:, 1] / N_SPLITS

    # LightGBM
    lgb_model = LGBMClassifier(**lgb_params)
    lgb_model.fit(X.iloc[tr], y.iloc[tr])

    oof[va, 1] = lgb_model.predict_proba(X.iloc[va])[:, 1]
    test_preds[:, 1] += lgb_model.predict_proba(test)[:, 1] / N_SPLITS

# -------- Meta Model --------
meta_model = LogisticRegression(max_iter=2000)
meta_model.fit(oof, y)

stack_oof = meta_model.predict_proba(oof)[:, 1]
stack_test = meta_model.predict_proba(test_preds)[:, 1]

print("FINAL STACKED OOF AUC:", roc_auc_score(y, stack_oof))


Fold 1
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 254208, number of negative: 312792
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.044672 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1846
[LightGBM] [Info] Number of data points in the train set: 567000, number of used features: 23
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.448339 -> initscore=-0.207386
[LightGBM] [Info] Start training from score -0.207386
Fold 2
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 254208, number of negative: 312792
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.128812 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1

In [10]:
# SUBMISSION
submission = pd.DataFrame({'id': test_ids, target: stack_test})
submission.to_csv('/kaggle/working/submission.csv', index=False)
print('submission.csv created')


submission.csv created


In [ ]:

# ================= LOGISTIC META STACKER =================

from sklearn.linear_model import LogisticRegression
import numpy as np

# Ensure xgb_oof, lgb_oof, xgb_test_preds, lgb_test_preds exist before running

meta_train = np.column_stack([xgb_oof, lgb_oof])
meta_test = np.column_stack([xgb_test_preds, lgb_test_preds])

meta_model = LogisticRegression(max_iter=1000)
meta_model.fit(meta_train, y)

final_test_preds = meta_model.predict_proba(meta_test)[:, 1]


In [ ]:

# ================= FINAL SUBMISSION =================

import pandas as pd

# Check expected submission format
try:
    sample = pd.read_csv("sample_submission.csv")
    print("Sample submission columns:", sample.columns.tolist())
except:
    print("sample_submission.csv not found, proceeding with default format.")

# Create submission
submission = pd.DataFrame({
    "id": test["id"],
    "Heart Disease": final_test_preds
})

submission.to_csv("submission.csv", index=False)

print("Submission file created: submission.csv")
submission.head()
